In [1]:
import kaleidocell
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import hashlib
from datetime import datetime

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad"
RESULTS_MP_PATH = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea_c7/results_mp.kc"
GSEA_CSV = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_6_gsea_c7_immune_all/04_gsea_c7_immunesigdb.csv"
GMT_PATH = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/c7.immunesigdb.v2026.1.Hs.symbols.gmt"
OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells"
os.makedirs(OUT_DIR, exist_ok=True)

In [9]:
# ── Step 1 — Load data and results ─────────────────────────────────────────
checkpoint("Loading data")
adata = sc.read_h5ad(DATA_PATH)
checkpoint(f"Data loaded: {adata.n_obs} cells, {adata.n_vars} genes")

checkpoint("Loading results_mp")
results_mp = kaleidocell.load(RESULTS_MP_PATH)
checkpoint(f"results_mp loaded: {len(results_mp['mp_dict'])} MPs")
mp_dict = results_mp["mp_dict"]

# Step 2 — Load significant terms────────────────────────────────────────────
gsea_df = pd.read_csv(GSEA_CSV)
checkpoint(f"Total significant terms: {len(gsea_df)}")

[2026-08-03 14:05:00] Loading data
[2026-08-03 14:05:45] Data loaded: 633710 cells, 25987 genes
[2026-08-03 14:05:45] Loading results_mp
[2026-08-03 14:05:47] results_mp loaded: 30 MPs
[2026-08-03 14:05:47] Total significant terms: 19542


In [11]:
import gseapy as gp
all_results = []

for mp_name, gene_scores in mp_dict.items():
    checkpoint(f"Running Enrichr for {mp_name}")
    gene_list = list(gene_scores.index.unique())

    try:
        enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets=GMT_PATH,
            outdir=None,
            no_plot=True,
        )
        res_df = enr.results.copy()
        res_df["MP"] = mp_name
        res_df["n_input_genes"] = len(gene_list)
        all_results.append(res_df)
        checkpoint(f"{mp_name}: {len(res_df)} pathways tested (n_genes={len(gene_list)})")
    except Exception as e:
        checkpoint(f"{mp_name} FAILED: {e}")

checkpoint("Combining all results")
full_gsea_results = pd.concat(all_results, ignore_index=True)
full_gsea_results.to_csv(os.path.join(OUT_DIR, "05_1_gsea_c7_immunesigdb_enrichr.csv"), index=False)
checkpoint("Saved full Enrichr results")

[2026-08-03 14:09:47] Running Enrichr for MP1
[2026-08-03 14:09:48] MP1: 2775 pathways tested (n_genes=55)
[2026-08-03 14:09:48] Running Enrichr for MP2
[2026-08-03 14:09:50] MP2: 2526 pathways tested (n_genes=41)
[2026-08-03 14:09:50] Running Enrichr for MP3
[2026-08-03 14:09:52] MP3: 488 pathways tested (n_genes=6)
[2026-08-03 14:09:52] Running Enrichr for MP4
[2026-08-03 14:09:53] MP4: 1336 pathways tested (n_genes=13)
[2026-08-03 14:09:53] Running Enrichr for MP5
[2026-08-03 14:09:55] MP5: 922 pathways tested (n_genes=8)
[2026-08-03 14:09:55] Running Enrichr for MP6
[2026-08-03 14:09:56] MP6: 3119 pathways tested (n_genes=60)
[2026-08-03 14:09:56] Running Enrichr for MP7
[2026-08-03 14:09:58] MP7: 1849 pathways tested (n_genes=28)
[2026-08-03 14:09:58] Running Enrichr for MP8
[2026-08-03 14:09:59] MP8: 2060 pathways tested (n_genes=30)
[2026-08-03 14:09:59] Running Enrichr for MP9
[2026-08-03 14:10:01] MP9: 1586 pathways tested (n_genes=26)
[2026-08-03 14:10:01] Running Enrichr for

In [14]:
checkpoint("Extracting top 10 terms per MP")

full_gsea_results["MP_num"] = full_gsea_results["MP"].str.extract(r"(\d+)").astype(int)

top10_per_mp = (
    full_gsea_results
    .sort_values(["MP_num", "Adjusted P-value"])
    .groupby("MP", sort=False)
    .head(10)
    .drop(columns="MP_num")
    .reset_index(drop=True)
)

top10_path = os.path.join(OUT_DIR, "05_1_gsea_c7_immunesigdb_enrichr_top10.csv")
top10_per_mp.to_csv(top10_path, index=False)
checkpoint(f"Saved top 10 terms per MP: {top10_path}")

[2026-08-03 14:17:24] Extracting top 10 terms per MP
[2026-08-03 14:17:24] Saved top 10 terms per MP: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_1_gsea_c7_immunesigdb_enrichr_top10.csv


In [3]:
full_gsea_results = pd.read_csv(os.path.join(OUT_DIR, "05_1_gsea_c7_immunesigdb_enrichr.csv"))

In [5]:
checkpoint("Filtering most significant terms per MP")

significant_results = full_gsea_results[full_gsea_results["Adjusted P-value"] < 0.05].copy()
significant_results["MP_num"] = significant_results["MP"].str.extract(r"(\d+)").astype(int)

top_significant = (
    significant_results
    .sort_values(["MP_num", "Adjusted P-value"])
    .groupby("MP", sort=False)
    .head(5)
    .drop(columns="MP_num")
    .reset_index(drop=True)
)

sig_path = os.path.join(OUT_DIR, "05_1_gsea_c7_immunesigdb_enrichr_top5_significant.csv")
top_significant.to_csv(sig_path, index=False)
checkpoint(f"Saved: {sig_path} ({top_significant['MP'].nunique()} MPs with signif. terms)")

[2026-08-03 15:10:16] Filtering most significant terms per MP
[2026-08-03 15:10:16] Saved: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_1_gsea_c7_immunesigdb_enrichr_top5_significant.csv (27 MPs with signif. terms)
